<span style="color: rgb(99, 100, 102); font-family: Roboto, sans-serif; background-color: rgb(255, 255, 255);"> Township is built upon farming and production puzzle cores, and casual order board games. The player is harvesting crops such as wheat, corn, carrot, potato, sugarcane, cocoa, tomato, rubber, silk, strawberries, rice and pepper. Assets are used to produce goods in factories to earn coins and experience points.

[Source](https://en.wikipedia.org/wiki/Township_(video_game))<span style="background-color: rgb(255, 255, 255);"><br></span>

See the first 1000 rows of items, the production time and the constraint they depend upon.

In [38]:
SELECT TOP (1000) 
      [portfolio].[township].[items].[Id] as itemId
      , [portfolio].[township].[items].[name] as itemName
      , [portfolio].[township].[items].[productiontime] as productionTime
      , [portfolio].[township].[constraints].name as constraintName
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  ORDER BY productionTime
  , constraintName; 
GO

(65 rows affected)

Total execution time: 00:00:00.078

itemId,itemName,productionTime,constraintName
1,gold,0,none
2,wheat,2,field
25,cow feed,4,feed mill
16,bread,5,bakery
3,corn,5,field
26,chicken feed,8,feed mill
4,carrot,10,field
29,cream,11,dairy factory
17,cookies,15,bakery
27,sheep feed,16,feed mill


List the items, their constraints, the production time and their dependancies.

In [39]:
SELECT TOP (1000) 
    [portfolio].[township].[items].[Id],
    [portfolio].[township].[items].[name] as [itemName],
    [portfolio].[township].[constraints].[name] AS [constraintName],
    [portfolio].[township].[items].[productiontime] as [productionTime],
    parentDetails.name AS [parentName],
    [portfolio].[township].[dependancies].[items] AS [numberOfItems]
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  LEFT JOIN [portfolio].[township].[dependancies] ON [portfolio].[township].[items].[Id] = [portfolio].[township].[dependancies].[itemId]
  JOIN (SELECT * FROM [portfolio].[township].[items]) as parentDetails ON [portfolio].[township].[dependancies].[parentId] = parentDetails.Id
  WHERE [portfolio].[township].[dependancies].[parentId] IS NULL

UNION ALL

SELECT TOP (1000) 
    [portfolio].[township].[items].[Id],
    [portfolio].[township].[items].[name] as [itemName],
    [portfolio].[township].[constraints].[name] AS [constraintName],
    [portfolio].[township].[items].[productiontime] as [productionTime],
    parentDetails.name AS [parentName],
    [portfolio].[township].[dependancies].[items] AS [numberOfItems]
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  LEFT JOIN [portfolio].[township].[dependancies] ON [portfolio].[township].[items].[Id] = [portfolio].[township].[dependancies].[itemId]
  JOIN (SELECT * FROM [portfolio].[township].[items]) as parentDetails ON [portfolio].[township].[dependancies].[parentId] = parentDetails.Id
  WHERE [portfolio].[township].[dependancies].[parentId] IS NOT NULL
  ORDER BY productionTime, itemName;
GO


(86 rows affected)

Total execution time: 00:00:00.062

Id,itemName,constraintName,productionTime,parentName,numberOfItems
2,wheat,field,2,gold,0
25,cow feed,feed mill,4,wheat,2
25,cow feed,feed mill,4,corn,1
16,bread,bakery,5,wheat,2
3,corn,field,5,gold,1
26,chicken feed,feed mill,8,wheat,2
26,chicken feed,feed mill,8,carrot,1
4,carrot,field,10,gold,2
29,cream,dairy factory,11,milk,1
17,cookies,bakery,15,wheat,2


list all the constraints and the dependent items ordered by production time.

In [40]:
SELECT TOP (1000) 
    [portfolio].[township].[constraints].[Id] as constriantId
    , [portfolio].[township].[constraints].[name] as constraintName
    ,[portfolio].[township].[items].[Id]
    ,[portfolio].[township].[items].[name] as itemName
    ,[portfolio].[township].[items].[productiontime] as productionTime
  FROM [portfolio].[township].[constraints]
  JOIN [portfolio].[township].[items] on [portfolio].[township].[constraints].[Id] = [portfolio].[township].[items].[constraintId]
  ORDER BY constriantId, productionTime, itemName;

(65 rows affected)

Total execution time: 00:00:00.017

constriantId,constraintName,Id,itemName,productionTime
1,apiary,24,honeycombs,360
2,bakery,16,bread,5
2,bakery,17,cookies,15
2,bakery,18,bagel,29
2,bakery,20,potato bread,57
2,bakery,19,pizza,114
3,candy factory,63,jelly beans,120
4,chicken coop,21,eggs,60
5,cowshed,22,milk,20
6,dairy factory,29,cream,11


List all posssible product combinations per constraint under the productuion time limit.

In [41]:
DECLARE @constraintId INT;
DECLARE @productionTimeLimit INT = 60;

-- Declare a cursor to iterate through each constraintId
DECLARE constraint_cursor CURSOR FOR
SELECT DISTINCT constraintId
FROM portfolio.township.items
WHERE constraintId NOT IN (1);

-- Open the cursor
OPEN constraint_cursor;

-- Fetch the first constraintId
FETCH NEXT FROM constraint_cursor INTO @constraintId;

-- Loop through each constraintId
WHILE @@FETCH_STATUS = 0
BEGIN
    -- Print the constraintId (for demonstration purposes)
    PRINT 'Processing constraintId: ' + CAST(@constraintId AS VARCHAR);
    -- Run the RecursiveCTE query for the current constraintId
    WITH RecursiveCTE AS (
    SELECT
        [portfolio].[township].[items].[constraintId]
        ,CAST([portfolio].[township].[items].[Id] AS VARCHAR(MAX)) AS Combination
        ,CAST([portfolio].[township].[items].[name] AS VARCHAR(MAX)) AS [description]
        ,[portfolio].[township].[items].[productiontime] AS totalProductionTime
    FROM [portfolio].[township].[items]
    WHERE constraintId = @constraintId 
    AND productiontime <= @productionTimeLimit

    UNION ALL

    SELECT
        i.constraintId
        ,rc.Combination + ',' + CAST(i.[Id] AS VARCHAR(MAX))
        ,rc.[description] + ',' + CAST(i.[name] AS VARCHAR(MAX))
        ,rc.totalProductionTime + i.productiontime
    FROM RecursiveCTE rc
    JOIN [portfolio].[township].[items] i ON i.Id > CAST(SUBSTRING(rc.Combination, LEN(rc.Combination) - CHARINDEX(',', REVERSE(rc.Combination)) + 2, LEN(rc.Combination)) AS INT)
    WHERE
        i.constraintId = @constraintId 
    AND
        rc.totalProductionTime + i.[productiontime] <= @productionTimeLimit
    AND NOT EXISTS (
            SELECT 1
            FROM [portfolio].[township].[items] ni
            WHERE ni.Id = i.Id
            AND ',' + rc.Combination + ',' LIKE '%,' + CAST(ni.Id AS VARCHAR(MAX)) + ',%'
        )
    )
    SELECT
        constraintId
        ,c.name
        ,Combination
        ,[description]
        ,totalProductionTime
    FROM
        RecursiveCTE
    JOIN 
        [portfolio].[township].[constraints] c ON c.Id = RecursiveCTE.constraintId
    ORDER BY
        constraintId, totalProductionTime DESC;
    -- Fetch the next constraintId
    FETCH NEXT FROM constraint_cursor INTO @constraintId;
END
-- Close and deallocate the cursor
CLOSE constraint_cursor;
DEALLOCATE constraint_cursor;

Processing constraintId: 2

(13 rows affected)

Processing constraintId: 3

(0 rows affected)

Processing constraintId: 4

...

constraintId,name,Combination,description,totalProductionTime
15,rubber factory,37,rubber,60


constraintId,name,Combination,description,totalProductionTime


constraintId,name,Combination,description,totalProductionTime
18,sugar factory,"34,33","syrup,sugar",60
18,sugar factory,"33,34","sugar,syrup",60
18,sugar factory,34,syrup,40
18,sugar factory,33,sugar,20


constraintId,name,Combination,description,totalProductionTime
19,textile factory,59,yarn,36
19,textile factory,58,cotton fabric,27


constraintId,name,Combination,description,totalProductionTime


list all the items, they dependancies, production time and constraints

In [42]:
SELECT TOP (1000) [itemId]
      ,[parentId]
      ,i.name AS itemName
      ,p.name AS parentName
      ,[items]
      ,i.productiontime as productionTime
      ,c.name AS constraintName
  FROM [portfolio].[township].[dependancies] d
  JOIN [portfolio].[township].[items] i on i.Id = d.itemId
  JOIN [portfolio].[township].[items] p on p.Id = d.parentId
  JOIN [portfolio].[township].[constraints] c on c.Id = i.constraintId
  order by itemName, parentName, productionTime

(86 rows affected)

Total execution time: 00:00:00.028

itemId,parentId,itemName,parentName,items,productionTime,constraintName
18,21,bagel,eggs,3,29,bakery
18,33,bagel,sugar,1,29,bakery
18,2,bagel,wheat,2,29,bakery
28,5,bee feed,sugarcane,1,24,feed mill
28,2,bee feed,wheat,3,24,feed mill
16,2,bread,wheat,2,5,bakery
42,31,brownie,butter,1,38,pastry factory
42,11,brownie,cacao,2,38,pastry factory
42,34,brownie,syrup,1,38,pastry factory
31,22,butter,milk,3,54,dairy factory


List all the parrents and their dependancies

In [43]:
WITH RecursiveCTE AS (
    SELECT 
        d.itemId
        ,d.parentId
        ,p.name AS parentName
        ,i.name AS itemName
        ,i.productiontime AS productionTime
        ,c.name AS constraintName
        ,d.items
        , 1 as [level]
    FROM 
        portfolio.township.dependancies d
    JOIN 
        portfolio.township.items i ON i.Id = d.itemId
    JOIN 
        portfolio.township.items p ON p.Id = d.parentId
    JOIN 
        portfolio.township.constraints c ON c.Id = i.constraintId
    WHERE 
        i.name = 'pizza'

    UNION ALL

    SELECT 
        d.itemId
        ,d.parentId
        ,p.name AS parentName
        ,i.name AS itemName
        ,i.productiontime AS productionTime
        ,c.name AS constraintName
        ,d.items
        , rc.[level]+1 as [level]
    FROM 
        portfolio.township.dependancies d
    JOIN 
        portfolio.township.items i ON i.Id = d.itemId
    JOIN 
        portfolio.township.items p ON p.Id = d.parentId
    JOIN 
        portfolio.township.constraints c ON c.Id = i.constraintId
    JOIN 
        RecursiveCTE rc ON rc.parentId = d.itemId
)
SELECT 
    *
FROM 
    RecursiveCTE
ORDER BY 
    [level], parentName, productionTime;

(11 rows affected)

Total execution time: 00:00:00.034

itemId,parentId,parentName,itemName,productionTime,constraintName,items,level
19,30,cheese,pizza,114,bakery,1,1
19,8,tomato,pizza,114,bakery,2,1
19,2,wheat,pizza,114,bakery,2,1
2,1,gold,wheat,2,field,0,2
8,1,gold,tomato,120,field,6,2
30,22,milk,cheese,27,dairy factory,2,2
22,25,cow feed,milk,20,cowshed,1,3
25,3,corn,cow feed,4,feed mill,1,4
25,2,wheat,cow feed,4,feed mill,2,4
2,1,gold,wheat,2,field,0,5


In [44]:
DECLARE @itemName VARCHAR(50);

-- Declare a cursor to iterate through each constraintId
DECLARE dependancy_cursor CURSOR FOR
SELECT DISTINCT [name]
FROM portfolio.township.items
WHERE [name] NOT IN ('pizza');

-- Open the cursor
OPEN dependancy_cursor;

-- Fetch the first constraintId
FETCH NEXT FROM dependancy_cursor INTO @itemName;

-- Loop through each @itemName
WHILE @@FETCH_STATUS = 0
BEGIN
    -- Print the constraintId (for demonstration purposes)
    PRINT 'Processing item: ' + CAST(@itemName AS VARCHAR);
    -- Run the RecursiveCTE query for the current @itemName
    WITH RecursiveCTE AS (
        SELECT 
            d.itemId
            ,d.parentId
            ,p.name AS parentName
            ,i.name AS itemName
            ,i.productiontime AS productionTime
            ,c.name AS constraintName
            ,d.items AS requiredItems
            , 1 as [level]
        FROM 
            portfolio.township.dependancies d
        JOIN 
            portfolio.township.items i ON i.Id = d.itemId
        JOIN 
            portfolio.township.items p ON p.Id = d.parentId
        JOIN 
            portfolio.township.constraints c ON c.Id = i.constraintId
        WHERE 
            i.name = @itemName

        UNION ALL

        SELECT 
            d.itemId
            ,d.parentId
            ,p.name AS parentName
            ,i.name AS itemName
            ,i.productiontime AS productionTime
            ,c.name AS constraintName
            ,d.items AS requiredItems
            , rc.[level]+1 as [level]
        FROM 
            portfolio.township.dependancies d
        JOIN 
            portfolio.township.items i ON i.Id = d.itemId
        JOIN 
            portfolio.township.items p ON p.Id = d.parentId
        JOIN 
            portfolio.township.constraints c ON c.Id = i.constraintId
        JOIN 
            RecursiveCTE rc ON rc.parentId = d.itemId
    )
    SELECT 
        itemId
        , itemName
        , parentId
        , parentName as ingredientsName
        , constraintName
        , requiredItems
        , productionTime
        , requiredItems * productionTime as totalProductionTime
        , requiredItems * (SELECT productiontime FROM portfolio.township.items itm where parentId = itm.Id) as totalProductionTime1
    FROM 
        RecursiveCTE
    ORDER BY 
        [level], parentName, productionTime;
    -- Fetch the next constraintId
    FETCH NEXT FROM dependancy_cursor INTO @itemName;
END
-- Close and deallocate the cursor
CLOSE dependancy_cursor;
DEALLOCATE dependancy_cursor;

Processing item: bagel

(11 rows affected)

Processing item: bee feed

(4 rows affected)

Processing item: bread

...

itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1
27,sheep feed,4,carrot,feed mill,2,16,32,20
27,sheep feed,3,corn,feed mill,2,16,32,10
3,corn,1,gold,field,1,5,5,0
4,carrot,1,gold,field,2,10,20,0


itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1
13,silk,1,gold,field,20,900,18000,0


itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1
60,silk fabric,13,silk,textile factory,2,81,162,1800
13,silk,1,gold,field,20,900,18000,0


itemId,itemName,parentId,ingredientsName,constraintName,requiredItems,productionTime,totalProductionTime,totalProductionTime1


: Msg 530, Level 16, State 1, Line 21
The statement terminated. The maximum recursion 100 has been exhausted before statement completion.